# Assignment 2: probe analysis

This notebook reads the output of your probe runs and scores it by the four
measures Assignment 2 is marked on: **syntax**, whether Qwen's answers decode;
**scene interpretation**, the hazards Qwen reports, hazard by hazard, as recall,
precision, and their F1; **SmolVLM's accepted answers**, the share the runtime
would act on; and **tactical decision**, whether the manoeuvre and speed proposed
fit the scene, before and after your arbiter. It also scores each answer's hazards
against the scene DriveBench measured, with `metadrive_starter.hazard_agreement`,
which the probe itself does not do, and reads closed-loop event logs with the
same module.

Its one task is your arbiter: it puts each answer through your
`submission/arbitration.py`, and ends with the structural gates' own checks of
it, printed as `PASS` or `NOT YET`.

It never runs MetaDrive or a model. It reads files your commands wrote. With no
runs of your own it makes an example: the two recorded semantic-failure answers,
replayed against every recorded probe scene.

The notebook is never submitted and never graded. The figures it prints come
from the files you give it, so cite them by the command that wrote those files,
never by this notebook.

Start it from the repository root:

```bash
uv run --extra notebooks jupyter lab course/assignment-2/probe-analysis.ipynb
```


In [ ]:
import json
import statistics
import tempfile
import time
from dataclasses import replace
from pathlib import Path

import matplotlib.image as mpimg
import matplotlib.pyplot as plt

from metadrive_starter.config import load_config
from metadrive_starter.env import control_timestep_s
from metadrive_starter.gates import CONTROL_LOOP_UNBLOCKED, MONOTONE_RESTRICTION
from metadrive_starter.hazard_agreement import (
    format_report,
    scan_event_logs,
    scene_hazards,
    score_hazards,
    summarize,
)
from metadrive_starter.recorded_probes import (
    INTERPRETATION_CATALOG,
    RECORDED_CATALOG,
    load_recorded_probes,
)
from metadrive_starter.replay import scene_from_payload
from metadrive_starter.submission import (
    PROJECT_ROOT,
    apply_submission_overlay,
    load_arbitration,
)
from metadrive_starter.vla.arbitration import (
    ArbitrationOutcome,
    ArbitrationRequest,
    ReviewRequest,
    arbitrate_assessment,
    review_command,
)
from metadrive_starter.vla.assessment import HazardType, RelativeLocation, RiskLevel, VLAHazard
from metadrive_starter.vla.provider import ModelFailureCategory
from metadrive_starter.vla.response import VLAResponseError, decode_vla_assessment
from metadrive_starter.vla_probe import replay_vla_probe_artifacts
from metadrive_starter.vla_runtime import build_vla_provider

SUBMISSION = PROJECT_ROOT / "submission"
# Your arbiter's settings are this configuration's command_validation values,
# after your agent.yaml. Every probe configuration sets the same ones.
ARBITER_CONFIG = PROJECT_ROOT / "configs" / "vla-probe-qwen3-vl-2b-llama-structured.yaml"
# Your probe output directories, one per `probe --submission ... --infer` run,
# relative to the repository root. Leave empty for the example.
PROBE_RUNS = []
# Your closed-loop event logs, or directories of them, from `run ... --event-log`.
EVENT_LOGS = []


def replay_example(config_name, output_root):
    """Replay one recorded answer to every recorded scene: no model, no MetaDrive."""
    config = load_config(PROJECT_ROOT / "configs" / config_name)
    fixture = replace(config.vla.fixture, path=str(PROJECT_ROOT / config.vla.fixture.path))
    provider = build_vla_provider(replace(config.vla, fixture=fixture))
    output = output_root / config_name.removesuffix(".yaml")
    replay_vla_probe_artifacts(config, RECORDED_CATALOG, output, provider=provider)
    return output


if not PROBE_RUNS:
    example_root = Path(tempfile.mkdtemp(prefix="probe-analysis-example-"))
    PROBE_RUNS = [
        replay_example(name, example_root)
        for name in (
            "demo-vla-semantic-stopped-vehicle.yaml",
            "demo-vla-semantic-partial-occlusion.yaml",
        )
    ]
    print(f"no runs given; replayed the example into {example_root}")


def load_run(directory):
    summary = json.loads((directory / "summary.json").read_text())
    scenes = {
        item["scenario_id"]: json.loads(
            (directory / item["scenario_id"] / "metadata.json").read_text()
        )
        for item in summary["scenarios"]
    }
    return {"name": directory.name, "dir": directory, "summary": summary, "scenes": scenes}


runs = [load_run(PROJECT_ROOT / Path(run)) for run in PROBE_RUNS]
for run in runs:
    print(f"{run['name']}: {len(run['scenes'])} scenes from {run['dir']}")

## Scoring against the measured scene

The probe checks each answer against what its scene's manifest expects: a
`vehicle` hazard `front`, say. Hazard agreement asks a different question:
does the answer invent a hazard the measured scene lacks, or leave out one in
the ego path? A scene can be scored that way only if it is one of the recorded
scenes, exactly as the manifest defines it; the probe renders the same scenes
every time, so the public catalog's always are.

In [ ]:
recorded = {
    probe.scenario_id: probe
    for probe in (*load_recorded_probes(), *load_recorded_probes(INTERPRETATION_CATALOG))
}
recorded_scenarios = {
    scenario_id: json.loads((probe.source_dir / "metadata.json").read_text())["scenario"]
    for scenario_id, probe in recorded.items()
}
# The decoder refused the answer; a timeout or transport failure got no answer.
DECODE_FAILURES = {
    VLAResponseError.__name__,
    *(error.__name__ for error in VLAResponseError.__subclasses__()),
}


def assessment_hazards(assessment):
    return [
        VLAHazard(
            hazard_type=HazardType(hazard["hazard_type"]),
            relative_location=RelativeLocation(hazard["relative_location"]),
            risk=RiskLevel(hazard["risk"]),
        )
        for hazard in assessment["relevant_hazards"]
    ]


def agreement(scenario_id, metadata):
    """The answer's hazards scored against the measured scene, or None."""
    assessment = metadata["outcome"].get("parsed_assessment")
    if assessment is None or recorded_scenarios.get(scenario_id) != metadata["scenario"]:
        return None
    return score_hazards(assessment_hazards(assessment), recorded[scenario_id].scene)


def mark(value):
    return {True: "pass", False: "FAIL", None: "-"}[value]


for run in runs:
    print(f"== {run['name']}")
    print(f"{'scene':>24}  {'answer':<44} {'perception':>10} {'agrees':>7} {'action':>7} {'speed':>6}")
    for scenario_id, metadata in run["scenes"].items():
        outcome = metadata["outcome"]
        evaluation = outcome.get("semantic_evaluation") or {}
        assessment = outcome.get("parsed_assessment")
        scored = agreement(scenario_id, metadata)
        if assessment is not None:
            answer = (
                f"{assessment['proposed_action']} at "
                f"{assessment['proposed_target_speed_mps']:.1f} m/s, "
                f"confidence {assessment['confidence']:.2f}"
            )
        else:
            answer = outcome.get("error_category") or outcome["status"]
        print(
            f"{scenario_id:>24}  {answer:<44} "
            f"{mark(evaluation.get('perception_passed')):>10} "
            f"{mark(None if scored is None else scored.agrees):>7} "
            f"{mark(evaluation.get('tactical_action_passed')):>7} "
            f"{mark(evaluation.get('raw_speed_passed')):>6}"
        )
    print()

`perception` is the probe's own check of the hazards the scene requires;
`agrees`, hazard agreement against the measured scene; `action`, whether the
proposed manoeuvre is one the scene accepts; `speed`, whether the proposed
target speed falls in the scene's range. The probe reports raw speed as a
diagnostic: the runtime maps each action to its own speed.

The `action` and `speed` columns judge an answer against what the manifest
expects of its scene. Your arbiter never knows that; it has the assessment and
the measured scene. So in the example, look at two rows the way an arbiter
would:

- On `stopped-vehicle`, the stopped-vehicle answer reports the car ahead
  correctly, so both hazard checks pass, and then proposes `KEEP_LANE` at cruise
  speed. Hazard agreement endorses it. Only checking the action and speed
  against the hazards it reported catches it: a high-risk vehicle in front does
  not go with keeping lane at cruise.
- On `partial-occlusion`, the other answer reports the truck as low risk, which
  satisfies the probe's perception check, and misses the stopped car behind it.
  Its fields agree with each other, a low-risk vehicle ahead and `KEEP_LANE`, so
  checking them against each other finds nothing. Only comparing its hazards
  with the measured scene catches it.

The two fail in opposite directions, and an arbiter needs both techniques.

## The criteria, per run

Each layer as scenes passed out of scenes scored. Syntax counts only answers
that arrived: a timeout is not a syntax failure.

In [ ]:
def layers(run):
    outcomes = [metadata["outcome"] for metadata in run["scenes"].values()]
    decoded = sum(outcome["status"] == "success" for outcome in outcomes)
    refused = sum(outcome.get("error_category") in DECODE_FAILURES for outcome in outcomes)
    scores = run["summary"]["scores"]
    agreements = [
        scored
        for scenario_id, metadata in run["scenes"].items()
        if (scored := agreement(scenario_id, metadata)) is not None
    ]
    return {
        "syntax: answers decoded": (decoded, decoded + refused),
        "scene interpretation: probe perception": (
            scores["perception"]["passed"],
            scores["perception"]["scored"],
        ),
        "scene interpretation: hazards agree": (
            sum(scored.agrees for scored in agreements),
            len(agreements),
        ),
        "tactical decision: probe action": (
            scores["tactical_action"]["passed"],
            scores["tactical_action"]["scored"],
        ),
        "tactical decision: raw speed": (
            scores["raw_speed"]["passed"],
            scores["raw_speed"]["scored"],
        ),
    }


run_layers = {run["name"]: layers(run) for run in runs}
names = list(next(iter(run_layers.values())))
print(f"{'':>42}" + "".join(f"{name[:22]:>24}" for name in run_layers))
for name in names:
    cells = "".join(
        f"{passed:>17} of {scored:<3}" for passed, scored in (run_layers[run][name] for run in run_layers)
    )
    print(f"{name:>42}{cells}")

## Scene interpretation, hazard by hazard

Scene interpretation is marked per hazard, on hidden scenes of its own, and a
scene's pass above is not that measure: it ignores an invented hazard. This cell
pools a run's answers as marking does, with `score_hazards`: each claim pairs
with one measured object of its type, at most one 45-degree sector away. **Recall**
is the share of the objects a claim paired with, **precision** the share of the
claims that paired, and **F1** their harmonic mean. An answer that does not decode
claims nothing and finds nothing, and a scene with no answer is left out.

The eight probe scenes hold only vehicles and one light. Marking also asks a model
to find pedestrians, cones, and barriers, so probe those too: the
[brief](README.md#probing-your-observation) gives the `probe --replay-from` command
for the eight recorded scenes that hold them, and each carries its truth, the
objects the camera shows. List that output directory in `PROBE_RUNS`.

`CONSTANT_ANSWER` is one answer scored as if the model gave it on every scene,
with no perception at all: your F1 means something only above the best such
answer. Change it, and see how much guessing alone earns on these scenes.


In [ ]:
# (type, relative_location) pairs, the same on every scene.
CONSTANT_ANSWER = [("vehicle", "front")]


def truth_of(scene_id):
    """The scene a claim pairs with, and the ids of the objects an answer must find.

    A scene recorded for scene interpretation keeps its truth beside it: the scene
    measured to 80 m, and which objects the camera shows within the request's 50 m.
    Any other scene has its measured scene, which may hold what the camera cannot show.
    """
    probe = recorded[scene_id]
    truth_file = probe.source_dir / "truth.json"
    if not truth_file.is_file():
        return probe.scene, {
            hazard.source_id for hazard in scene_hazards(probe.scene) if hazard.required
        }
    scene = scene_from_payload(json.loads((probe.source_dir / "truth_scene.json").read_text()))
    return scene, {
        item["object_id"]
        for item in json.loads(truth_file.read_text())["objects"]
        if item["in_view"] and item["within_detection_radius"]
    }


def hazard_counts(hazards, scene_id):
    """(found, required, paired, claims) for one answer on one recorded scene."""
    scene, required = truth_of(scene_id)
    scored = score_hazards(hazards, scene)
    found = sum(hazard.source_id in required for hazard in scored.matched)
    return found, len(required), len(scored.claims) - len(scored.false_claims), len(scored.claims)


def pooled(counts):
    """Recall, precision, and F1 over every answer's counts at once."""
    found, required, paired, claims = (sum(column) for column in zip(*counts))
    recall = found / required if required else 0.0
    precision = paired / claims if claims else 0.0
    f1 = 0.0 if recall + precision == 0.0 else 2 * recall * precision / (recall + precision)
    return {
        "recall": recall,
        "precision": precision,
        "f1": f1,
        "found": found,
        "required": required,
        "paired": paired,
        "claims": claims,
    }


def scored_scenes(run):
    """A run's recorded scenes with an answer: {scene id: hazards it claimed}."""
    claimed = {}
    for scenario_id, metadata in run["scenes"].items():
        if recorded_scenarios.get(scenario_id) != metadata["scenario"]:
            continue
        outcome = metadata["outcome"]
        if outcome["status"] == "success":
            claimed[scenario_id] = assessment_hazards(outcome["parsed_assessment"])
        elif outcome.get("error_category") in DECODE_FAILURES:
            claimed[scenario_id] = []
    return claimed


def show_interpretation(name, figures):
    print(
        f"{name:>42}: F1 {figures['f1']:.3f}, recall {figures['recall']:.3f} "
        f"({figures['found']}/{figures['required']}), precision {figures['precision']:.3f} "
        f"({figures['paired']}/{figures['claims']})"
    )


constant_hazards = [
    VLAHazard(HazardType(hazard), RelativeLocation(location), RiskLevel.MEDIUM)
    for hazard, location in CONSTANT_ANSWER
]
for run in runs:
    claimed = scored_scenes(run)
    print(f"== {run['name']}: {len(claimed)} scenes")
    if not claimed:
        continue
    show_interpretation(
        "the run's answers",
        pooled([hazard_counts(hazards, scene) for scene, hazards in claimed.items()]),
    )
    show_interpretation(
        "the constant answer",
        pooled([hazard_counts(constant_hazards, scene) for scene in claimed]),
    )


## SmolVLM's accepted answers

Below the safety floor's `minimum_confidence` the runtime discards a command, and
the probe does not: an answer that decodes shows as a `success` here, whatever its
confidence. This cell counts what marking counts, the share of a run's SmolVLM
answers that decode at or above the SmolVLM profile's own floor, which your
`agent.yaml` does not move. Grading asks with structured output on, so read runs
of `configs/vla-probe-smolvlm-llama-structured.yaml`. A scene with no answer is
left out.


In [ ]:
SMOLVLM_CONFIG = PROJECT_ROOT / "configs" / "vla-probe-smolvlm-llama-structured.yaml"
smolvlm_floor = load_config(SMOLVLM_CONFIG).command_validation.minimum_confidence


def smolvlm_answers(run):
    """Every answered request of one SmolVLM run: its outcomes, or [] for another model."""
    outcomes = [metadata["outcome"] for metadata in run["scenes"].values()]
    if not any("smolvlm" in (outcome.get("model_id") or "").lower() for outcome in outcomes):
        return []
    return [
        outcome
        for outcome in outcomes
        if outcome["status"] == "success" or outcome.get("error_category") in DECODE_FAILURES
    ]


def accepted(answers):
    """(accepted, decoded): how many decode at or above the floor, and how many decode."""
    confidences = [
        outcome["parsed_assessment"]["confidence"]
        for outcome in answers
        if outcome["status"] == "success"
    ]
    return sum(confidence >= smolvlm_floor for confidence in confidences), len(confidences)


asked_smolvlm = False
for run in runs:
    answers = smolvlm_answers(run)
    if answers:
        asked_smolvlm = True
        kept, decoded = accepted(answers)
        print(
            f"{run['name']}: {kept} of {len(answers)} answers accepted ({kept / len(answers):.3f}); "
            f"{decoded} decoded, {decoded - kept} of them below the confidence floor "
            f"{smolvlm_floor:g}"
        )
if not asked_smolvlm:
    print(
        "None of these runs asked SmolVLM. Probe it, for example:\n"
        "  uv run metadrive-starter probe --config configs/vla-probe-smolvlm-llama-structured.yaml "
        "--submission submission --infer --seed 1 --output-dir tmp/probe-smolvlm-structured-01\n"
        "then list the directory in PROBE_RUNS and rerun the notebook."
    )


## What the model received, and what it answered

One scene of one run, side by side: the image the probe sent, the model's raw
answer, and how its hazards compare with the measured scene. Change `RUN` and
`SCENE`.

In [ ]:
RUN = runs[-1]
SCENE = "partial-occlusion"
# A run of the recorded interpretation scenes holds none of the probe's scenes.
if SCENE not in RUN["scenes"]:
    SCENE = next(iter(RUN["scenes"]))

scene_dir = RUN["dir"] / SCENE
metadata = RUN["scenes"][SCENE]
plt.figure(figsize=(9, 5))
plt.imshow(mpimg.imread(scene_dir / "frame.png"))
plt.title(f"{RUN['name']} / {SCENE}: the image sent")
plt.axis("off")
plt.show()

raw_response = scene_dir / "raw_response.txt"
print("answered:", raw_response.read_text() if raw_response.is_file() else "nothing")
scored = agreement(SCENE, metadata)
if scored is not None:
    print()
    for hazard in scored.scene_hazards:
        found = "reported" if hazard in scored.matched else "missed"
        path = "in the ego path" if hazard.in_path else "off the ego path"
        print(
            f"measured {hazard.hazard_type.value} at {hazard.relative_location.value}, "
            f"{hazard.distance_m:.1f} m, {path}: {found}"
        )
    for claim in scored.false_claims:
        print(f"reported {claim.hazard_type.value} at {claim.relative_location.value}: not in the scene")

## Confidence

Does the model's confidence tell good answers from bad? Every answer from every
run, by whether its action passed. The models measured so far gave nearly one
value whatever the scene, so confidence has carried no measurable signal. Test
that claim on your own runs before your arbiter relies on it, and close it as
an empty investigation, with evidence, if it holds.

In [ ]:
plt.figure(figsize=(8, 3))
for offset, run in enumerate(runs):
    points = [
        (
            metadata["outcome"]["parsed_assessment"]["confidence"],
            metadata["outcome"]["semantic_evaluation"]["tactical_action_passed"],
        )
        for metadata in run["scenes"].values()
        if metadata["outcome"].get("semantic_evaluation")
    ]
    plt.scatter(
        [confidence for confidence, _ in points],
        [float(passed) + 0.08 * offset for _, passed in points],
        label=run["name"],
        alpha=0.7,
    )
plt.yticks([0, 1], ["action failed", "action passed"])
plt.xlim(-0.05, 1.05)
plt.xlabel("confidence")
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

## Your arbiter on these answers

**Task: your arbiter.** At the `TODO` lines in `submission/arbitration.py`,
decide how far to trust each answer in `arbitrate`, with the two techniques
above. Then rerun the notebook from the top.

Each decoded answer on a recorded scene goes through your arbiter as the
tactical-decision mark puts it: a fresh episode, `begin_episode()` and then
`arbitrate`, with the scene measured now (the recorded scene is still, so only
its time moves) and the answer aged as grading ages it, by the token counts the
probe recorded (as late as the model took, where it recorded none). `review`,
which Assignment 2 leaves returning True,
then sees the command your arbiter let through, once on a tick without a failure
and once with each failure a model request can report; each line ends with what
it did.

The cell ends with the structural gates' own checks of your arbiter, printed as
`PASS` or `NOT YET`. They tell you whether every answer could stand, never
whether it was the right one: the tactical-decision mark measures that.

In [ ]:
arbiter_config = apply_submission_overlay(load_config(ARBITER_CONFIG), SUBMISSION)
settings = arbiter_config.command_validation
Arbiter = load_arbitration(SUBMISSION)
CONTROL_STEP_S = control_timestep_s(arbiter_config.simulator)
# A tick without a failure, then each failure a model request can report.
FAILURES = (None, *ModelFailureCategory)


def answer_age_s(outcome):
    """How late an answer reaches your arbiter at grading: by its token counts, as
    the brief's tactical-decision paragraph states, never by the clock."""
    tokens = outcome.get("provider_metadata") or {}
    prompt, answer = tokens.get("input_tokens"), tokens.get("output_tokens")
    if prompt is None or answer is None:
        # This server reported no counts: as late as the model took.
        return outcome["latency_s"]
    return 0.34 + 0.00018 * prompt + 0.005 * answer


def arbitrate_run(run):
    """Every decoded answer of one run on a recorded scene, through your arbiter."""
    arbiter = Arbiter(settings)
    rows = []
    for scenario_id, metadata in run["scenes"].items():
        if (
            metadata["outcome"]["status"] != "success"
            or recorded_scenarios.get(scenario_id) != metadata["scenario"]
        ):
            continue
        raw_response = run["dir"] / scenario_id / "raw_response.txt"
        issued_at_s = metadata["capture"]["timestamp_s"]
        now_s = issued_at_s + answer_age_s(metadata["outcome"])
        # The scene your arbiter gets was measured now, as the live loop hands
        # it one: the recorded scene is still, so only its time moves.
        scene = replace(recorded[scenario_id].scene, timestamp_s=now_s)
        cruise_speed_mps = metadata["capture"]["cruise_speed_mps"]
        started_s = time.perf_counter()
        arbiter.begin_episode()
        begun_s = time.perf_counter() - started_s
        assessment = decode_vla_assessment(raw_response.read_text())
        decision = arbitrate_assessment(
            arbiter,
            ArbitrationRequest(
                assessment=assessment,
                scene=scene,
                now_s=now_s,
                issued_at_s=issued_at_s,
                cruise_speed_mps=cruise_speed_mps,
            ),
        )
        # Below minimum_confidence the safety floor falls back, as it does on a decline.
        holds = (
            decision.outcome is not ArbitrationOutcome.DECLINED
            and decision.assessment.confidence >= settings.minimum_confidence
        )
        reviews = {
            failure: review_command(
                arbiter,
                ReviewRequest(
                    active=decision.assessment if holds else None,
                    issued_at_s=issued_at_s if holds else None,
                    scene=scene,
                    now_s=now_s,
                    cruise_speed_mps=cruise_speed_mps,
                    failure=failure,
                ),
            )
            for failure in FAILURES
        }
        rows.append(
            {
                "scene": scenario_id,
                "assessment": assessment,
                "decision": decision,
                "holds": holds,
                "reviews": reviews,
                "slowest_s": max(
                    begun_s,
                    decision.elapsed_s,
                    *(review.elapsed_s for review in reviews.values()),
                ),
            }
        )
    return rows


def describe_decision(row):
    decision = row["decision"]
    if decision.problem is not None:
        return "declined for you: your answer could not stand"
    if decision.outcome is ArbitrationOutcome.DECLINED:
        return "declined"
    text = decision.outcome.value
    if decision.outcome is ArbitrationOutcome.RESTRICTED:
        text += (
            f" to {decision.assessment.proposed_target_speed_mps:.1f} m/s, "
            f"confidence {decision.assessment.confidence:.2f}"
        )
    if not row["holds"]:
        text += ", below minimum_confidence: the floor falls back"
    return text


def describe_review(row):
    if not row["holds"]:
        return "no command holds authority to review"
    ended = [
        "no failure" if failure is None else failure.value
        for failure, review in row["reviews"].items()
        if review.revoked
    ]
    return "review ends it on " + ", ".join(ended) if ended else "review keeps it"


def arbiter_checks(arbitrated):
    """The structural gates' checks of your arbiter: (gate, passed, detail)."""
    rows = [row for _, run_rows in arbitrated for row in run_rows]
    problems = [
        f"{run['name']} / {row['scene']}: {problem}"
        for run, run_rows in arbitrated
        for row in run_rows
        for problem in (
            row["decision"].problem,
            *(review.problem for review in row["reviews"].values()),
        )
        if problem is not None
    ]
    if problems:
        monotone = (
            False,
            f"{len(problems)} answers could not stand; the first: {problems[0]}",
        )
    else:
        reviewed = sum(len(row["reviews"]) for row in rows)
        monotone = (
            True,
            f"{len(rows)} assessments arbitrated and {reviewed} ticks reviewed, "
            "none granting more authority than the model proposed",
        )
    slowest_s = max(row["slowest_s"] for row in rows)
    if slowest_s <= CONTROL_STEP_S:
        unblocked = (
            True,
            f"every arbiter call returned within the {CONTROL_STEP_S:g} s control step, "
            f"the slowest in {1000.0 * slowest_s:.2f} ms",
        )
    else:
        unblocked = (
            False,
            f"the slowest arbiter call took {1000.0 * slowest_s:.2f} ms, longer than "
            f"the {CONTROL_STEP_S:g} s control step",
        )
    return [(MONOTONE_RESTRICTION, *monotone), (CONTROL_LOOP_UNBLOCKED, *unblocked)]


arbitrated = [(run, arbitrate_run(run)) for run in runs]
for run, rows in arbitrated:
    print(f"== {run['name']}")
    for row in rows:
        proposal = row["assessment"]
        print(
            f"{row['scene']:>24}  {proposal.proposed_action.value} at "
            f"{proposal.proposed_target_speed_mps:.1f} m/s, confidence {proposal.confidence:.2f}: "
            f"{describe_decision(row)}; {describe_review(row)}"
        )
    print()
if any(rows for _, rows in arbitrated):
    for gate, passed, detail in arbiter_checks(arbitrated):
        print(f"{'PASS   ' if passed else 'NOT YET'}  {gate}: {detail}")
else:
    print("No decoded answer on a recorded scene: nothing for your arbiter to arbitrate.")

## Reruns: the median and the spread

A live model answers differently under each sampling seed. Grading reruns every
hidden item five times, each under its own seed, and reports the median and the
variance, so a submission that is stable across reruns does better than one that
sampled well once. Repeat one probe command into new directories, each under a
new `--seed`, as the brief describes; under one seed a rerun repeats the last.
List them all in `PROBE_RUNS`, and see how
far each layer moves. Runs pool only when they sent the same observations to the
same model: the cell groups them that way, gives a median only within a group,
and reports each group apart, to compare, never to pool. The example's two runs
are two groups.

In [ ]:
def identity(run):
    """What a run sent and to whom: reruns share it."""
    prompts, models = set(), set()
    for metadata in run["scenes"].values():
        prompts.add(metadata["capture"]["prompt_sha256"])
        outcome = metadata["outcome"]
        provider = outcome.get("provider_metadata") or {}
        models.add((outcome.get("model_id"), provider.get("fixture_id")))
    return frozenset(prompts), frozenset(models)


def rerun_groups(runs):
    """The runs grouped by what they sent and to whom: only reruns share a group."""
    groups = {}
    for run in runs:
        groups.setdefault(identity(run), []).append(run)
    return list(groups.values())


groups = rerun_groups(runs)
if len(groups) > 1:
    print(
        f"These runs differ in observation or model, so they form {len(groups)} groups, "
        "each reported apart: compare them, never pool them.\n"
    )
for group in groups:
    print("== " + ", ".join(run["name"] for run in group))
    group_layers = [layers(run) for run in group]
    for name in names:
        shares = [
            passed / scored
            for passed, scored in (run_layer[name] for run_layer in group_layers)
            if scored
        ]
        if len(shares) > 1:
            print(
                f"{name:>42}: median {statistics.median(shares):.2f}, "
                f"range {min(shares):.2f} to {max(shares):.2f}, "
                f"variance {statistics.pvariance(shares):.3f}, over {len(shares)} runs"
            )
        elif shares:
            print(f"{name:>42}: {shares[0]:.2f}, in one run")
    print()

## Closed-loop event logs

The probe asks once per scene. A closed-loop run asks again and again as the
scene changes, and every `command_validation` event in its event log holds the
model's assessment beside the scene it was validated against. The same module
scores them all, per model profile, with confidence binned against agreement.
Answers replayed from fixtures are reported apart from live model calls, and
each fixture apart from every other.

In [ ]:
if EVENT_LOGS:
    report = summarize(scan_event_logs(PROJECT_ROOT / Path(path) for path in EVENT_LOGS))
    print(format_report(report))
else:
    print(
        "No event logs given. Write one with a closed-loop run, for example:\n"
        "  uv run metadrive-starter run --config configs/demo-vla-semantic-stopped-vehicle.yaml "
        "--submission submission --event-log tmp/semantic-stopped-vehicle-01.jsonl\n"
        "then list it in EVENT_LOGS and rerun this cell."
    )